# Ozaki scheme II: INT8 matmul

From [this paper](https://arxiv.org/abs/2504.08009). FP64-accurate matmul from INT8 tensor cores.

Requires T4/RTX/A100+ (INT8 tensor cores). Run on [Colab with T4 runtime](https://colab.research.google.com/).

| Step | Operation | Types |
|------|-----------|-------|
| 1 | scale rows of $A$, cols of $B$ | float64 -> int64 |
| 2 | reduce mod primes $p_i < 128$ | int64 -> int8 |
| 3 | matmul per prime (tensor cores) | int8 × int8 -> int32 |
| 4 | CRT reconstruction | int32 -> float64 |
| 5 | rescale | float64 |

Steps 3-5 are one fused Triton kernel.

In [ ]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("git clone https://github.com/PritRaj1/tensor_inv.git 2>/dev/null")
    os.system("pip install -e /content/tensor_inv")
else:
    src = "src" if os.path.isdir("src") else "../src"
    sys.path.insert(0, src)

In [2]:
import torch
import math
import time

from tensor_inv import ozaki_matmul

## CRT

Given coprime $m_1, \ldots, m_L$ and residues $r_i = x \bmod m_i$, CRT recovers $x$ uniquely if $|x| < \prod m_i / 2$.

| Inner dim $K$ | Bits needed | Primes |
|---------------|-------------|--------|
| 64 | $\sim 111$ | ~20 |
| 256 | $\sim 113$ | ~21 |
| 1024 | $\sim 115$ | ~25 |
| 4096 | $\sim 117$ | ~26 |

In [3]:
primes = [3, 5, 7, 11, 13]
M = math.prod(primes)
x = 12345
residues = [x % p for p in primes]

x_rec = 0
for r, m in zip(residues, primes):
    Mi = M // m
    x_rec += r * Mi * pow(Mi, -1, m)
x_rec %= M

print(f"x = {x}, residues = {residues}")
print(f"recovered = {x_rec}, match: {x == x_rec}")

x = 12345, residues = [0, 0, 4, 3, 8]
recovered = 12345, match: True


## Accuracy

In [4]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

sizes = [32, 64, 128, 256, 512]
if device == "cpu":
    sizes = [32, 64, 128]

print(f"device: {device}")
print(f"{'n':>6} {'max_abs_err':>14} {'max_rel_err':>14}")
print("-" * 38)

for n in sizes:
    A = torch.randn(n, n, dtype=torch.float64, device=device)
    B = torch.randn(n, n, dtype=torch.float64, device=device)

    C_ozaki = ozaki_matmul(A, B)
    C_ref = A @ B

    abs_err = (C_ozaki - C_ref).abs().max().item()
    nz = C_ref.abs() > 1e-12
    rel_err = ((C_ozaki - C_ref).abs()[nz] / C_ref.abs()[nz]).max().item()

    print(f"{n:>6} {abs_err:>14.3e} {rel_err:>14.3e}")

device: cuda
     n    max_abs_err    max_rel_err
--------------------------------------


/home/pritmanguy/Work/tensor_inv/src/tensor_inv/_triton_matmul.py:74:27: error: 'arith.extf' op operand #0 must be floating-point-like, but got 'tensor<32x32xi8, #ttg.dot_op<{opIdx = 0, parent = #ttg.blocked<{sizePerThread = [2, 2], threadsPerWarp = [2, 16], warpsPerCTA = [4, 1], order = [1, 0]}>}>>'
            c += tl.dot(a, b)
                          ^
module {
  tt.func public @_fused_kernel(%arg0: !tt.ptr<i8> {tt.divisibility = 16 : i32}, %arg1: !tt.ptr<i8> {tt.divisibility = 16 : i32}, %arg2: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg3: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg4: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg5: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg6: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg7: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg8: !tt.ptr<f64> {tt.divisibility = 16 : i32}, %arg9: !tt.ptr<i32> {tt.divisibility = 16 : i32}, %arg10: i32 {tt.divisibility = 16 : i32}, %arg11: i32 {tt.divisibility = 16 : i32}, %arg12: i32 {tt.d

RuntimeError: PassManager::run failed

## Benchmark

`ozaki_matmul` vs `torch.matmul` (FP64). INT8 tensor cores have 16-32x raw throughput over FP64.

In [ ]:
def bench(fn, A, B, warmup=5, runs=20):
    for _ in range(warmup):
        fn(A, B)
    if A.is_cuda:
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(runs):
        fn(A, B)
    if A.is_cuda:
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs


if device == "cpu":
    print("benchmark requires CUDA, skipping")
else:
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    bench_sizes = [64, 128, 256, 512, 1024, 2048]
    print(f"{'n':>6} {'torch (ms)':>12} {'ozaki (ms)':>12} {'ratio':>8}")
    print("-" * 42)

    for n in bench_sizes:
        A = torch.randn(n, n, dtype=torch.float64, device=device)
        B = torch.randn(n, n, dtype=torch.float64, device=device)

        t_torch = bench(torch.matmul, A, B) * 1000
        t_ozaki = bench(ozaki_matmul, A, B) * 1000

        print(f"{n:>6} {t_torch:>12.2f} {t_ozaki:>12.2f} {t_ozaki / t_torch:>7.2f}x")